In [1]:
#Python
#import re
#import random
import importlib
from collections import Counter
import unicodedata
#Terceros
import pandas as pd
import numpy as np
#Este proyecto
import set_paths
import config.paths as pths 
from config.patrones import Pattern, PATRONES, abrev_dict
from src.replace_abrev import tokenizar_direccion, reemplazar_abrevs 
from src.elim_noise import limpiar_ruido
from src.cleaning_pipe import clean


## TEST REEMPLAZO ABREVIATURAS

In [2]:
#Test
direcciones = pd.read_csv(pths.DIRECCIONES_CLEAN / "universo_direcciones_clean.csv")
direcciones_test = direcciones["DIRECCIÓN GARANTÍA"].sample(n=10, random_state=1105)
for direccion in direcciones_test:
    tokens = tokenizar_direccion(direccion)
    reemplazadas = reemplazar_abrevs(tokens, abrev_dict)
    print(f"{direccion}")
    print(reemplazadas)

CALLE FLAMBOYAN LOTE 17 MANZANA 001 SM 002
['Calle', 'FLAMBOYAN', 'Lt.', '17', 'Mz.', '001', 'SM', '002']
MANZANA 53 LOTE 198 CALLE RET  8 CIELO CASA D
['Mz.', '53', 'Lt.', '198', 'Calle', 'RET', '8', 'CIELO', 'CASA', 'D']
CALLE BETELGUESE NO 106 LT 23 MZ 05
['Calle', 'BETELGUESE', ' No.', '106', 'Lt.', '23', 'Mz.', '05']
Av. Hidalgo , Dpto. 101, Ed. 6 No. 1
['Av.', 'Hidalgo', 'Dpto.', '101', 'Ed.', '6', 'No.', '1']
CERRADA DEL RAYO  OFICIAL 29-A LOTE 29 LA VILLITA
['CERRADA', 'DEL', 'RAYO', 'OFICIAL', '29', '-', 'A', 'Lt.', '29', 'LA', 'VILLITA']
Torre Jacarandas, Condominio 178, Conjunto Condominal Viva Mx, Centlapal identif
['Torre', 'Jacarandas', 'Condominio', '178', 'Conjunto', 'Condominal', 'Viva', 'Mx', 'Centlapal', 'identif']
PRIVADA LAS FLORES, NUM EXT 2, M-6, L-6
['PRIVADA', 'LAS', 'FLORES', 'No.', 'Ext.', '2', 'Mz.', '-', '6', 'Lt.', '-', '6']
MZ 51 LTE 2 C. HDA LA JALAPILLA N NUM 15
['Mz.', '51', 'Lt.', '2', 'Calle', 'HDA', 'LA', 'JALAPILLA', ' No.', 'No.', '15']
BOULEVARD 

## TEST N-GRAMAS

In [3]:
direcciones["tokens"] = direcciones["DIRECCIÓN GARANTÍA"].astype(str).apply(lambda x:reemplazar_abrevs(tokenizar_direccion(x), abrev_dict))

In [4]:
counters = {
    1: Counter(),
    2: Counter(),
    3: Counter(),
    4: Counter(),
    5: Counter(),
    8: Counter()
}

for tokens in direcciones["tokens"]:
    for n, counter in counters.items():
        for i in range(len(tokens)-n+1):
            counter[tuple(tokens[i:i+n])] += 1 # -> Construir N-gramas 
#counters[5].most_common() -> Patrones más comunes por longitud (5)

## TEST ENCONTRAR FRASES

In [5]:
np.random.seed(1105)
test_adress = np.random.randint(0, direcciones.shape[0]-1)
print(direcciones["DIRECCIÓN GARANTÍA"][test_adress])
print(limpiar_ruido(tokens = direcciones["tokens"][test_adress]))
print("")

3RA CDA FELIPE ANGELES MZA A LT 9
['3', 'RA', 'CDA', 'FELIPE', 'ANGELES', 'Mz.', 'A', 'Lt.', '9']



In [6]:
for i in range(30):
    #np.random.seed(int(np.log(1105)+i))
    test_adress = np.random.randint(0, direcciones.shape[0]-1)
    print(direcciones["DIRECCIÓN GARANTÍA"][test_adress])
    print(clean(raw_adress = direcciones["DIRECCIÓN GARANTÍA"][test_adress]))
    #time.sleep(10)

PRIV GUELAGUETZA MZ 27 LT 1 C 31 SMZ 237 RDICIAL M
['PRIV', 'GUELAGUETZA', 'Mz.', '27', 'Lt.', '1', 'C', '31', 'SMZ', '237', 'RDICIAL', 'Mz.']
SAN MARTIN CABALLERO 4 INT 27     LT 4 MZ 12
['SAN', 'MARTIN', 'CABALLERO', '4', 'Int.', '27', 'Lt.', '4', 'Mz.', '12']
C LAZARO CARDENAS SN
['C', 'LAZARO', 'CARDENAS', 'SN']
ANDRES PEREZ DE RIBAS NORTE  165
['ANDRES', 'PEREZ', 'DE', 'RIBAS', 'NORTE', '165']
SIN NOMBRE SN
['SIN', 'NOMBRE', 'SN']
MZ 79 LTE 36 C. VALLE PASOS VIV. NUM A
['Mz.', '79', 'Lt.', '36', 'Calle', 'VALLE', 'PASOS', 'VIV.', 'No.', 'A']
Bosque Santa Catalina  Agua Prieta Centro 84200
['Bosque', 'Santa', 'Catalina', 'Agua', 'Prieta', 'Centro', '84200']
U 4 EDIF 10 0 1005 JARDIN BALBUENA
['U', '4', 'EDIF', '10', '0', '1005', 'JARDIN', 'BALBUENA']
SEXTO CERRADA DE BOSQUES DE MEXICO, NUM EXT 4, M-106, L-22
['SEXTO', 'CERRADA', 'DE', 'BOSQUES', 'DE', 'MEXICO', 'No.', 'Ext.', '4', 'Mz.', '106', 'Lt.', '22']
CARRETERA FEDERAL KM 78 MEXICO-PUEBLA #439 LT. XOMETITLA
['CARRETERA', 'FED